<a href="https://colab.research.google.com/github/SheffieldML/GPc/blob/main/python/examples/expquad_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ExpQuad regression with `gpc` (CIP-0024)

GPy's lengthscale-first squared exponential is `ExpQuad`. In `gpc`, `kernel="eq"` (aliases `ExpQuad` / `expquad`) builds `CEqKern` so paths are `ExpQuad.*` instead of `rbf.*`.

GPy (conceptual)::

```python
k = GPy.kern.ExpQuad(1) + GPy.kern.Bias(1) + GPy.kern.White(1)
m = GPy.models.GPRegression(X, Y, k)
m.ExpQuad.lengthscale = 0.2
m.optimize()
```

`gpc` (signal-only string; noise on likelihood)::

```python
m = gpc.GPRegression(X, Y, kernel="eq")
m.set_by_path("ExpQuad.lengthscale", 0.2)
```

Same \(K_of_r\) as RBF; storage remains `inverseWidth` under the hood.

```bash
pip install matplotlib ipywidgets
```


In [ ]:
%pip install --no-cache-dir --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "gpc==0.1.0a5"


In [ ]:
%pip install matplotlib ipywidgets


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

rng = np.random.default_rng(0)
X = np.linspace(0.0, 1.0, 20)[:, None]
Y = np.sin(2 * np.pi * X) + 0.05 * rng.normal(size=X.shape)

m = gpc.GPRegression(X, Y, kernel="eq")
m.set_by_path("ExpQuad.lengthscale", 0.2)
m.set_by_path("ExpQuad.variance", 1.0)
print("paths:", m.param_paths())
print("objective before:", m.objective())
m.optimize(iters=50, messages=True)
print("objective after:", m.objective())
mu, var = m.predict(X)
print("predict shapes:", mu.shape, var.shape, "mean var", float(var.mean()))
print(m)


In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

# Side-by-side: ExpQuad paths vs RBF paths (same covariance family).
m_rbf = gpc.GPRegression(X, Y, kernel="rbf")
m_rbf.set_by_path("rbf.lengthscale", 0.2)
m_rbf.optimize(iters=50)

Xtest = np.linspace(-0.1, 1.1, 200)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m, X, Y, Xtest=Xtest, ax=axes[0], label="ExpQuad")
axes[0].set_title("kernel='eq' → ExpQuad.*")
plot_1d(m_rbf, X, Y, Xtest=Xtest, ax=axes[1], label="RBF")
axes[1].set_title("kernel='rbf' → rbf.*")
plt.tight_layout()
plt.show()
